# 02 — UIT Training (Paper-faithful, Colab A100 80GB high-RAM)

Train UIT (Unified Image-Text) model theo paper [Hybrid, Unified and Iterative](https://dl.acm.org/doi/10.1145/3701716.3717850) (WWW 2025).

**Method (faithful):**

- Image encoder: Swin-B (`swin_base_patch4_window7_224_22k`)
- Text encoder: BERT-base-uncased
- Losses: `L = L_itc + L_itm + L_mlm + 0.1356 · L_mim`
- Hyperparams: AdamW lr=1e-5, wd=0.05, cosine + 5% warmup, **batch 256** (paper 84, A100-80GB cho phép tăng), 224×224, **22 epoch**
- Pretrained init: paper's `pretrained.pth` hoặc paper-released `lhp_2/beit3/checkpoint/...` (LHP-trained checkpoint)

**Inputs:**
- `manifests/train_manifest_trainable.parquet` (95% sau val split)
- `INPUT_ROOT/annotation/train/imgs_*.json` (75 files JSONL)
- `INPUT_ROOT/train/imgs_*/...` (1M images)

**Outputs:**
- `checkpoints/uit/checkpoint_best.pth` + per-epoch `checkpoint_<ep>.pth`
- `checkpoints/uit/training_log.json`
- Auto-sync mỗi epoch sang Drive

**ETA:** ~8h trên A100 80GB BF16 (paper-equivalent 22 epoch, batch 256 thay vì 84).

In [ ]:
# === Bootstrap & paths ===
from pathlib import Path
import os, sys, json, shutil, subprocess, warnings
warnings.filterwarnings('ignore')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

NOTEBOOK_DIR = Path('.').resolve()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from aic_colab_utils import (
    setup_aic2026_environment, select_a100_device,
    sync_chunk_to_drive, wait_for_pending_syncs, maybe_clear_cuda,
)

PATHS = setup_aic2026_environment()
INPUT_ROOT = PATHS['input_root']
MANIFEST_DIR = PATHS['manifest_dir']
LOCAL_ROOT = PATHS['local_root']
OUTPUT_ROOT = PATHS['output_root']
DRIVE_OUTPUT_ROOT = PATHS['drive_output_root']

UIT_CKPT_DIR = OUTPUT_ROOT / 'checkpoints' / 'uit'
UIT_CKPT_DIR.mkdir(parents=True, exist_ok=True)

# Code clone target
UIT_REPO = LOCAL_ROOT / 'aio_repo'
UIT_CODE = UIT_REPO / 'uit' / 'cmp'

device = select_a100_device()
print('UIT_CKPT_DIR:', UIT_CKPT_DIR)
print('UIT_CODE will be at:', UIT_CODE)

In [ ]:
# === Clone AIO paper repo (or rsync from Drive cache) ===
#
# Repo `Hybrid-Unified-and-Iterative-A-Novel-Framework-for-Text-based-Person-Anomaly-Retrieval`
# lưu sẵn trong workspace local; trên Colab cần upload qua Drive hoặc clone từ GitHub.
DRIVE_AIO_REPO = Path('/content/drive/MyDrive/aic2026_data/aio_repo')
WORKSPACE_AIO_REPO = Path('/home/bao/Documents/AIC2026/Hybrid-Unified-and-Iterative-A-Novel-Framework-for-Text-based-Person-Anomaly-Retrieval')

if not UIT_REPO.exists():
    if DRIVE_AIO_REPO.exists():
        print(f'rsync {DRIVE_AIO_REPO} → {UIT_REPO}')
        subprocess.check_call(['rsync', '-a', f'{DRIVE_AIO_REPO}/', f'{UIT_REPO}/'])
    elif WORKSPACE_AIO_REPO.exists():
        print(f'rsync {WORKSPACE_AIO_REPO} → {UIT_REPO}')
        subprocess.check_call(['rsync', '-a', f'{WORKSPACE_AIO_REPO}/', f'{UIT_REPO}/'])
    else:
        print('Repo not found. Please upload to Drive at:', DRIVE_AIO_REPO)
        raise SystemExit('AIO repo missing')

assert UIT_CODE.exists(), f'{UIT_CODE} not found after rsync'
print('UIT code ready at:', UIT_CODE)

In [ ]:
# === Install paper-specific deps (compatible w/ Colab torch 2.x) ===
#
# Paper requires torch 2.5, timm 0.6.13, transformers 4.47.1, torchscale 0.3.0.
# Colab usually has torch ≥ 2.3; we pin compatible subset.
def pip_install(*pkgs):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *pkgs])

try:
    import timm, torchscale, transformers, yacs  # noqa: F401
except Exception:
    pip_install(
        'timm==0.6.13', 'torchscale==0.3.0', 'transformers==4.47.1',
        'tokenizers>=0.21.0', 'yacs', 'ruamel.yaml', 'tensorboardX',
        'pyarrow', 'pandas', 'einops',
    )
import pandas as pd, numpy as np, torch
print('torch:', torch.__version__, 'cuda:', torch.cuda.is_available())

In [ ]:
# === Stage data into paper-expected layout ===
#
# Paper expects:
#   data/PAB/annotation/train/imgs_*.json
#   data/PAB/train/imgs_*/{goal,full,wentwrong}/*.jpg
#   data/PAB/name-masked_test-set/{gallery, query_text.json, ...}
#
# Our INPUT_ROOT already has this layout (sau từ Kaggle). Tạo symlink từ code dir.
DATA_ROOT = UIT_REPO / 'data'
DATA_ROOT.mkdir(exist_ok=True)
PAB_LINK = DATA_ROOT / 'PAB'
if PAB_LINK.exists() and not PAB_LINK.is_symlink():
    shutil.rmtree(PAB_LINK)
if not PAB_LINK.exists():
    # Symlink INPUT_ROOT (chuẩn: chuỗi path của Kaggle PAB dataset) → data/PAB
    os.symlink(str(INPUT_ROOT), str(PAB_LINK))
print('Symlink:', PAB_LINK, '→', os.readlink(PAB_LINK))

# === Generate train_file list (locại bỏ val_split + val_zeroshot_scene rows) ===
#
# Paper-default uses all 75 imgs_*.json. We need to substitute custom JSONL
# that excludes val rows. Strategy: write `data/PAB/annotation/train_trainable/imgs_*.json`
# only with rows in train_manifest_trainable.parquet.
from collections import defaultdict

trainable_df = pd.read_parquet(MANIFEST_DIR / 'train_manifest_trainable.parquet')
print(f'train_trainable: {len(trainable_df):,} rows')

# Group by source annotation_file
groups = defaultdict(list)
for row in trainable_df.itertuples(index=False):
    groups[row.annotation_file].append({
        'image': row.annotation_image,
        'caption': row.caption,
        'image_id': row.image_id,
        'scene': row.scene,
        ('anomaly' if row.label_type == 'anomaly' else 'normal'): row.action,
    })

TRAINABLE_ANN_DIR = INPUT_ROOT / 'annotation' / 'train_trainable'
TRAINABLE_ANN_DIR.mkdir(parents=True, exist_ok=True)
imgs_jsons = sorted(groups.keys(), key=lambda s: int(s.split('_')[1].split('.')[0]))
for name in imgs_jsons:
    out = TRAINABLE_ANN_DIR / name
    with open(out, 'w', encoding='utf-8') as f:
        for item in groups[name]:
            f.write(json.dumps(item, ensure_ascii=False) + '\n')
print(f'Wrote {len(imgs_jsons)} trainable JSONL files → {TRAINABLE_ANN_DIR}')

In [ ]:
# === Generate runtime cmp.yaml for A100-80GB ===
#
# Override paper defaults:
#  - batch_size_train: 84 → 256 (A100-80GB)
#  - lr: 1e-4 → 1e-5 (paper-suggested for fine-tune)  
#  - epochs: 30 → 22 (paper-faithful)
#  - train_file: point to trainable JSONL (excludes val rows)
import yaml
from pathlib import Path as _P

BASE_CFG = UIT_CODE / 'configs' / 'cmp.yaml'
with open(BASE_CFG) as f:
    cfg = yaml.safe_load(f)

# Replace train_file paths
TRAINABLE_REL = '../../data/PAB/annotation/train_trainable'
train_files = []
for j in sorted(_P(TRAINABLE_ANN_DIR).glob('imgs_*.json'), key=lambda p: int(p.stem.split('_')[1])):
    train_files.append(f'{TRAINABLE_REL}/{j.name}')
cfg['train_file'] = train_files

# A100-80GB tuning
cfg['batch_size_train'] = 256
cfg['batch_size_test'] = 256
cfg['batch_size_test_text'] = 256
cfg['optimizer']['lr'] = 1.0e-5
cfg['optimizer']['weight_decay'] = 0.05
cfg['schedular']['lr'] = 1.0e-5
cfg['schedular']['epochs'] = 22
cfg['schedular']['num_warmup_steps'] = 1000  # ~5% of 22 epoch x ~3.8k iter
cfg['schedular']['sched'] = 'cosine'

RUNTIME_CFG = UIT_CODE / 'configs' / 'cmp_a100_80gb.yaml'
with open(RUNTIME_CFG, 'w') as f:
    yaml.safe_dump(cfg, f, sort_keys=False)
print('Wrote runtime config:', RUNTIME_CFG)
print('batch_size_train:', cfg['batch_size_train'])
print('epochs:', cfg['schedular']['epochs'])
print('train_files:', len(cfg['train_file']))

In [ ]:
# === Restore latest checkpoint from Drive (resume logic) ===
#
# Drive lưu mỗi epoch checkpoint tại drive_output_root/checkpoints/uit/.
# Restore latest checkpoint_*.pth vào output_dir nếu có.
import re

UIT_OUTPUT_DIR = UIT_CODE / 'out' / 'cmp_a100_80gb'
UIT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

drive_ckpts = (DRIVE_OUTPUT_ROOT / 'checkpoints' / 'uit')
drive_ckpts.mkdir(parents=True, exist_ok=True)
latest_drive = None
for p in drive_ckpts.glob('checkpoint_*.pth'):
    m = re.match(r'checkpoint_(\d+)\.pth', p.name)
    if m:
        if latest_drive is None or int(m.group(1)) > int(re.match(r'checkpoint_(\d+)\.pth', latest_drive.name).group(1)):
            latest_drive = p

if latest_drive is not None:
    dst = UIT_OUTPUT_DIR / latest_drive.name
    if not dst.exists():
        shutil.copy2(latest_drive, dst)
    print(f'Resume from {dst.name} ({int(re.match(r"checkpoint_(\d+)\.pth", dst.name).group(1))})')
else:
    print('No prior UIT checkpoint on Drive — fresh training run.')

# Init checkpoint from paper-released LHP-trained weights (or paper's pretrained.pth)
INIT_CKPT_DRIVE = Path('/content/drive/MyDrive/aic2026_data/pretrained/uit_pretrained.pth')
INIT_CKPT_LOCAL = UIT_REPO / 'checkpoint' / 'pretrained.pth'
if not INIT_CKPT_LOCAL.exists():
    INIT_CKPT_LOCAL.parent.mkdir(parents=True, exist_ok=True)
    if INIT_CKPT_DRIVE.exists():
        shutil.copy2(INIT_CKPT_DRIVE, INIT_CKPT_LOCAL)
        print(f'Init ckpt restored → {INIT_CKPT_LOCAL}')
    else:
        print(f'⚠️  No init checkpoint at {INIT_CKPT_DRIVE} — train from scratch (Swin-B + BERT init).')
else:
    print(f'Init ckpt present: {INIT_CKPT_LOCAL}')

In [ ]:
# === Run training (subprocess) + per-epoch Drive sync hook ===
#
# Paper's Search.py expects to be launched via torch.distributed.run.
# We patch Search.py at runtime to add a post-epoch hook that syncs
# checkpoint to Drive. Alternative: poll output_dir in a background thread.
import threading, time

def _drive_sync_thread(stop_event, out_dir, drive_dir, poll_sec=60):
    seen = set()
    while not stop_event.is_set():
        for p in Path(out_dir).glob('checkpoint_*.pth'):
            if p.name in seen:
                continue
            try:
                shutil.copy2(p, drive_dir / p.name)
                seen.add(p.name)
                print(f'[drive-sync] {p.name} → {drive_dir}')
            except Exception as e:
                print(f'[drive-sync] {p.name} failed: {e}')
        # Also sync log.txt periodically
        log_p = Path(out_dir) / 'log.txt'
        if log_p.exists():
            try:
                shutil.copy2(log_p, drive_dir / 'log.txt')
            except Exception:
                pass
        stop_event.wait(poll_sec)

stop_evt = threading.Event()
sync_t = threading.Thread(
    target=_drive_sync_thread,
    args=(stop_evt, UIT_OUTPUT_DIR, drive_ckpts),
    daemon=True,
)
sync_t.start()

# Launch training
env = os.environ.copy()
env['CUDA_VISIBLE_DEVICES'] = '0'
env['PYTHONUNBUFFERED'] = '1'

cmd = [
    sys.executable, '-m', 'torch.distributed.run',
    '--nproc_per_node=1', '--nnodes=1', '--node_rank=0',
    '--master_addr=127.0.0.1', '--master_port=16668',
    'Search.py',
    '--config', 'configs/cmp_a100_80gb.yaml',
    '--task', 'cmp',
    '--output_dir', str(UIT_OUTPUT_DIR),
    '--checkpoint', str(INIT_CKPT_LOCAL) if INIT_CKPT_LOCAL.exists() else '',
    '--bs', '256',
    '--epo', '22',
    '--lr', '1e-5',
    '--seed', '20260514',
]
print('Launch:', ' '.join(cmd))
proc = subprocess.Popen(cmd, cwd=str(UIT_CODE), env=env)
try:
    proc.wait()
finally:
    stop_evt.set()
    sync_t.join(timeout=120)

# Final sync best checkpoint
best = UIT_OUTPUT_DIR / 'checkpoint_best.pth'
if best.exists():
    final_dst = UIT_CKPT_DIR / 'checkpoint_best.pth'
    shutil.copy2(best, final_dst)
    shutil.copy2(best, drive_ckpts / 'checkpoint_best.pth')
    print(f'Best checkpoint synced → {final_dst}')
else:
    print('⚠️  No checkpoint_best.pth produced. Check Search.py output.')

wait_for_pending_syncs()
print('UIT training done.')

In [ ]:
# === Save training log meta ===
import time
meta = {
    'completed_at': time.strftime('%Y-%m-%d %H:%M:%S'),
    'config': 'cmp_a100_80gb.yaml',
    'batch_size': 256,
    'epochs': 22,
    'lr': 1.0e-5,
    'seed': 20260514,
    'uit_output_dir': str(UIT_OUTPUT_DIR),
    'best_checkpoint': str(UIT_CKPT_DIR / 'checkpoint_best.pth'),
}
with open(UIT_CKPT_DIR / 'training_log.json', 'w') as f:
    json.dump(meta, f, indent=2)
print(json.dumps(meta, indent=2))